# Parametric PINN Training Notebook

This notebook trains a parametric PINN for 1D heat conduction with conditioning variables `mu = [T0, q_type, q_amp, q_freq]`.


## Setup


In [1]:
import sys
from pathlib import Path
from dataclasses import dataclass
import json
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "src").exists():
    raise RuntimeError("Could not find project root containing /src")
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.utils import set_seed, get_device
from src.data import load_manifest_rows, load_case_manifest_row
from src.pinn import MLP, LossWeights

set_seed(42)
device = get_device()
print(f"Project root: {ROOT}")
print(f"Device: {device}")

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
OUTDIR = ROOT / "outputs" / "parametric" / RUN_ID
OUTDIR.mkdir(parents=True, exist_ok=True)
print(f"RUN_ID: {RUN_ID}")
print(f"OUTDIR: {OUTDIR}")


Project root: c:\Users\wscm13\OneDrive - Loughborough University\Part C\IDP\Individual Project\PINN
Device: cpu
RUN_ID: 20260309_144024
OUTDIR: c:\Users\wscm13\OneDrive - Loughborough University\Part C\IDP\Individual Project\PINN\outputs\parametric\20260309_144024


## Data Loading and Case Split


In [2]:
manifest = ROOT / "data" / "manifest.csv"
rows = load_manifest_rows(manifest)
if len(rows) < 2:
    raise RuntimeError("Need at least two cases for train/held-out split.")

# Deterministic split with at least one full held-out case.
rows_sorted = sorted(rows, key=lambda r: r["case_id"])
n_val_cases = max(1, int(round(0.2 * len(rows_sorted))))
val_rows = rows_sorted[-n_val_cases:]
train_rows = rows_sorted[:-n_val_cases]

if len(train_rows) == 0:
    raise RuntimeError("Split produced no training cases.")

print("Train cases:", [r["case_id"] for r in train_rows])
print("Held-out cases:", [r["case_id"] for r in val_rows])


Train cases: ['const_10000', 'const_15000', 'const_5000', 'offset_sine_1', 'sine_A10000_T100', 'sine_A10000_T50']
Held-out cases: ['sine_A5000_T100', 'sine_A5000_T50']


## Data Preprocessing and Parametric Batch


In [ ]:
def _extract_mu_raw(row: dict) -> dict:
    q_type_raw = str(row["q_right_type"]).lower()
    params = json.loads(str(row["q_right_params"]))

    T0 = float(row["T_left"])
    if q_type_raw == "constant":
        q_amp = float(params.get("A", 0.0))
        q_freq = 0.0
    else:
        q_amp = float(params.get("A", params.get("q0", 0.0)))
        omega = float(params.get("omega", 0.0))
        q_freq = omega / (2.0 * np.pi)

    return {
        "T0": T0,
        "q_type_raw": q_type_raw,
        "q_amp": q_amp,
        "q_freq": q_freq,
    }


def compute_mu_stats(cases_raw):
    eps = 1e-8
    T0_all = np.array([c["mu_raw"]["T0"] for c in cases_raw], dtype=np.float32)
    q_amp_all = np.array([c["mu_raw"]["q_amp"] for c in cases_raw], dtype=np.float32)
    q_freq_all = np.array([c["mu_raw"]["q_freq"] for c in cases_raw], dtype=np.float32)

    return {
        "T0_min": float(np.min(T0_all)),
        "T0_max": float(np.max(T0_all)),
        "q_amp_min": float(np.min(q_amp_all)),
        "q_amp_max": float(np.max(q_amp_all)),
        "q_freq_min": float(np.min(q_freq_all)),
        "q_freq_max": float(np.max(q_freq_all)),
        "eps": eps,
    }


def normalise_mu(mu_raw, mu_stats, case_physical=None) -> np.ndarray:
    eps = float(mu_stats.get("eps", 1e-8))

    if case_physical is not None and ("T_ref" in case_physical) and ("dT_ref" in case_physical):
        T_ref = float(case_physical["T_ref"])
        dT_ref = float(case_physical["dT_ref"])
        if abs(dT_ref) < eps:
            theta0 = 0.0
        else:
            theta0 = (float(mu_raw["T0"]) - T_ref) / dT_ref
    else:
        T0_min = float(mu_stats["T0_min"])
        T0_max = float(mu_stats["T0_max"])
        theta0 = 2.0 * (float(mu_raw["T0"]) - T0_min) / (T0_max - T0_min + eps) - 1.0

    q_type = str(mu_raw["q_type_raw"]).lower()
    is_const = 1.0 if q_type == "constant" else 0.0
    is_sine = 0.0 if q_type == "constant" else 1.0

    q_amp_min = float(mu_stats["q_amp_min"])
    q_amp_max = float(mu_stats["q_amp_max"])
    q_freq_min = float(mu_stats["q_freq_min"])
    q_freq_max = float(mu_stats["q_freq_max"])

    q_amp_norm = 2.0 * (float(mu_raw["q_amp"]) - q_amp_min) / (q_amp_max - q_amp_min + eps) - 1.0
    q_freq_norm = 2.0 * (float(mu_raw["q_freq"]) - q_freq_min) / (q_freq_max - q_freq_min + eps) - 1.0

    return np.array([theta0, is_const, is_sine, q_amp_norm, q_freq_norm], dtype=np.float32)


def _load_case_with_mu(row: dict) -> dict:
    c = load_case_manifest_row(row, root=ROOT)
    c["mu_raw"] = _extract_mu_raw(row)
    L = float(c["physical"]["L"])
    alpha = float(c["physical"]["alpha"])
    k = float(c["physical"]["k"])
    dT_ref = float(c["physical"]["dT_ref"])
    c["time_scale"] = np.float32((L**2) / alpha)         # t = tau * time_scale
    c["flux_scale"] = np.float32(k * dT_ref / L)          # theta_x = -q / flux_scale
    return c


train_cases = [_load_case_with_mu(r) for r in train_rows]
val_cases = [_load_case_with_mu(r) for r in val_rows]

mu_stats = compute_mu_stats(train_cases + val_cases)
for c in (train_cases + val_cases):
    c["mu"] = normalise_mu(c["mu_raw"], mu_stats, case_physical=c.get("physical"))

print(f"Loaded {len(train_cases)} train cases and {len(val_cases)} held-out cases.")
print("mu_stats:", {k: round(v, 6) for k, v in mu_stats.items() if k != "eps"})

const_examples = [c for c in (train_cases + val_cases) if c["mu_raw"]["q_type_raw"] == "constant"]
sine_examples = [c for c in (train_cases + val_cases) if c["mu_raw"]["q_type_raw"] != "constant"]
if const_examples:
    print("Example mu (constant):", const_examples[0]["mu"])
if sine_examples:
    print("Example mu (sine):", sine_examples[0]["mu"])

all_mu = np.stack([c["mu"] for c in (train_cases + val_cases)], axis=0)
print("mu_vector mins:", all_mu.min(axis=0))
print("mu_vector maxs:", all_mu.max(axis=0))


@dataclass
class ParamPINNBatch:
    xi_r: torch.Tensor
    tau_r: torch.Tensor
    mu_r: torch.Tensor

    xi_ic: torch.Tensor
    tau_ic: torch.Tensor
    mu_ic: torch.Tensor
    theta_ic: torch.Tensor

    xi_bc: torch.Tensor
    tau_bc: torch.Tensor
    mu_bc: torch.Tensor
    bc_time_scale: torch.Tensor
    bc_flux_scale: torch.Tensor

    xi_data: torch.Tensor
    tau_data: torch.Tensor
    mu_data: torch.Tensor
    theta_data: torch.Tensor


def _repeat_mu(mu_vec: np.ndarray, n: int) -> np.ndarray:
    return np.repeat(mu_vec.reshape(1, -1), n, axis=0).astype(np.float32)


def build_parametric_batch(
    cases,
    device,
    n_r_per_case=6000,
    data_fraction=0.25,
    seed=42,
    full_data=False,
):
    rng = np.random.default_rng(seed)

    xi_r_list, tau_r_list, mu_r_list = [], [], []
    xi_ic_list, tau_ic_list, mu_ic_list, theta_ic_list = [], [], [], []
    xi_bc_list, tau_bc_list, mu_bc_list, bc_ts_list, bc_fs_list = [], [], [], [], []
    xi_d_list, tau_d_list, mu_d_list, theta_d_list = [], [], [], []

    for case in cases:
        mu = case["mu"]
        tau_max = float(np.max(case["nondim"]["tau"]))

        # Collocation: (x, t, mu)
        xi_r = rng.uniform(0.0, 1.0, size=(n_r_per_case, 1)).astype(np.float32)
        tau_r = rng.uniform(0.0, tau_max, size=(n_r_per_case, 1)).astype(np.float32)
        mu_r = _repeat_mu(mu, n_r_per_case)

        # IC: (x, 0, mu)
        xi_ic = case["ic"]["xi_ic"].astype(np.float32)
        tau_ic = case["ic"]["tau_ic"].astype(np.float32)
        theta_ic = case["ic"]["theta_ic"].astype(np.float32)
        mu_ic = _repeat_mu(mu, xi_ic.shape[0])

        # BC: (x_boundary, t, mu)
        xi_bc = case["bc"]["xi_bc"].astype(np.float32)
        tau_bc = case["bc"]["tau_bc"].astype(np.float32)
        mu_bc = _repeat_mu(mu, xi_bc.shape[0])

        # Data: (x, t, mu, T_ref/theta_ref)
        xi_all = case["interior"]["xi_data_all"].astype(np.float32)
        tau_all = case["interior"]["tau_data_all"].astype(np.float32)
        theta_all = case["interior"]["theta_data_all"].astype(np.float32)

        n_all = xi_all.shape[0]
        if full_data:
            keep_idx = np.arange(n_all)
        else:
            n_keep = max(1, int(data_fraction * n_all))
            keep_idx = rng.choice(n_all, size=n_keep, replace=False)

        xi_data = xi_all[keep_idx]
        tau_data = tau_all[keep_idx]
        theta_data = theta_all[keep_idx]
        mu_data = _repeat_mu(mu, xi_data.shape[0])

        xi_r_list.append(xi_r); tau_r_list.append(tau_r); mu_r_list.append(mu_r)
        xi_ic_list.append(xi_ic); tau_ic_list.append(tau_ic); mu_ic_list.append(mu_ic); theta_ic_list.append(theta_ic)
        xi_bc_list.append(xi_bc); tau_bc_list.append(tau_bc); mu_bc_list.append(mu_bc)
        bc_ts_list.append(bc_time_scale); bc_fs_list.append(bc_flux_scale)
        xi_d_list.append(xi_data); tau_d_list.append(tau_data); mu_d_list.append(mu_data); theta_d_list.append(theta_data)

    def tt(arrs):
        return torch.tensor(np.vstack(arrs), dtype=torch.float32, device=device)

    return ParamPINNBatch(
        xi_r=tt(xi_r_list), tau_r=tt(tau_r_list), mu_r=tt(mu_r_list),
        xi_ic=tt(xi_ic_list), tau_ic=tt(tau_ic_list), mu_ic=tt(mu_ic_list), theta_ic=tt(theta_ic_list),
        xi_bc=tt(xi_bc_list), tau_bc=tt(tau_bc_list), mu_bc=tt(mu_bc_list),
        bc_time_scale=tt(bc_ts_list), bc_flux_scale=tt(bc_fs_list),
        xi_data=tt(xi_d_list), tau_data=tt(tau_d_list), mu_data=tt(mu_d_list), theta_data=tt(theta_d_list),
    )


def build_parametric_batches(
    cases,
    device,
    n_r_per_case=6000,
    data_fraction=0.25,
    seed=42,
    full_data=False,
):
    batches = []
    for i, case in enumerate(cases):
        batches.append(
            build_parametric_batch(
                [case],
                device=device,
                n_r_per_case=n_r_per_case,
                data_fraction=data_fraction,
                seed=int(seed + i),
                full_data=full_data,
            )
        )
    return batches


SHORT_TEST = False
n_r_per_case = 1200 if SHORT_TEST else 6000
train_batch = build_parametric_batch(
    train_cases,
    device=device,
    n_r_per_case=n_r_per_case,
    data_fraction=0.08 if SHORT_TEST else 0.25,
    seed=42,
    full_data=False,
)
val_batch = build_parametric_batch(
    val_cases,
    device=device,
    n_r_per_case=max(200, n_r_per_case // 4),
    data_fraction=1.0,
    seed=123,
    full_data=True,
)
val_batches = build_parametric_batches(
    val_cases,
    device=device,
    n_r_per_case=max(200, n_r_per_case // 4),
    data_fraction=1.0,
    seed=123,
    full_data=True,
)

train_case_probe = build_parametric_batch(
    [train_cases[0]],
    device=device,
    n_r_per_case=n_r_per_case,
    data_fraction=0.08 if SHORT_TEST else 0.25,
    seed=42,
    full_data=False,
)

print("Train samples (stacked compatibility batch):")
print("  collocation:", int(train_batch.xi_r.shape[0]))
print("  ic:", int(train_batch.xi_ic.shape[0]))
print("  bc:", int(train_batch.xi_bc.shape[0]))
print("  data:", int(train_batch.xi_data.shape[0]))
print("Train samples per case-step (probe):")
print("  collocation:", int(train_case_probe.xi_r.shape[0]))
print("  ic:", int(train_case_probe.xi_ic.shape[0]))
print("  bc:", int(train_case_probe.xi_bc.shape[0]))
print("  data:", int(train_case_probe.xi_data.shape[0]))
print("Val samples (all val cases stacked):")
print("  collocation:", int(val_batch.xi_r.shape[0]))
print("  ic:", int(val_batch.xi_ic.shape[0]))
print("  bc:", int(val_batch.xi_bc.shape[0]))
print("  data:", int(val_batch.xi_data.shape[0]))


Loaded 6 train cases and 2 held-out cases.
mu_stats: {'T0_min': 293.149994, 'T0_max': 293.149994, 'q_amp_min': 5000.0, 'q_amp_max': 15000.0, 'q_freq_min': 0.0, 'q_freq_max': 0.02}
Example mu (constant): [ 0.0000000e+00  1.0000000e+00  0.0000000e+00 -1.0000889e-12
 -1.0000000e+00]
Example mu (sine): [ 0.         0.         1.        -1.         0.2499994]
mu_vector mins: [ 0.  0.  0. -1. -1.]
mu_vector maxs: [0.         1.         1.         1.         0.99999905]
Train samples (stacked compatibility batch):
  collocation: 36000
  ic: 306
  bc: 6006
  data: 76572
Train samples per case-step (probe):
  collocation: 6000
  ic: 51
  bc: 1001
  data: 12762
Val samples (all val cases stacked):
  collocation: 3000
  ic: 102
  bc: 2002
  data: 102102


## Sanity & Verification

Fast checks to validate parametric PINN wiring before long training.


In [4]:
import random
import warnings

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
print(f"Verification seed set to {SEED}")


Verification seed set to 42


In [5]:
# 1) Shape + dtype + device checks

def _assert_batch_shapes_and_types(batch, name, expected_mu_dim=4):
    def chk_tensor(t, tname, ncols):
        assert isinstance(t, torch.Tensor), f"{name}.{tname} is not a torch.Tensor"
        assert t.ndim == 2 and t.shape[1] == ncols, f"{name}.{tname} expected shape (N,{ncols}), got {tuple(t.shape)}"
        assert torch.is_floating_point(t), f"{name}.{tname} must be floating type, got {t.dtype}"
        assert t.dtype == torch.float32, f"{name}.{tname} expected float32, got {t.dtype}"
        assert t.device == device, f"{name}.{tname} expected device {device}, got {t.device}"

    chk_tensor(batch.xi_r, "xi_r", 1)
    chk_tensor(batch.tau_r, "tau_r", 1)
    chk_tensor(batch.mu_r, "mu_r", expected_mu_dim)

    chk_tensor(batch.xi_ic, "xi_ic", 1)
    chk_tensor(batch.tau_ic, "tau_ic", 1)
    chk_tensor(batch.mu_ic, "mu_ic", expected_mu_dim)
    chk_tensor(batch.theta_ic, "theta_ic", 1)

    chk_tensor(batch.xi_bc, "xi_bc", 1)
    chk_tensor(batch.tau_bc, "tau_bc", 1)
    chk_tensor(batch.mu_bc, "mu_bc", expected_mu_dim)

    chk_tensor(batch.xi_data, "xi_data", 1)
    chk_tensor(batch.tau_data, "tau_data", 1)
    chk_tensor(batch.mu_data, "mu_data", expected_mu_dim)
    chk_tensor(batch.theta_data, "theta_data", 1)

_assert_batch_shapes_and_types(train_batch, "train_batch")
_assert_batch_shapes_and_types(val_batch, "val_batch")

m = train_batch.mu_r.shape[1]
N_check = min(16, int(train_batch.xi_r.shape[0]))
X_check = torch.cat([train_batch.xi_r[:N_check], train_batch.tau_r[:N_check], train_batch.mu_r[:N_check]], dim=1)
assert X_check.shape == (N_check, 2 + m), f"Model input expected {(N_check, 2+m)}, got {tuple(X_check.shape)}"

model_shape_probe = MLP(in_dim=2 + m, hidden=20, layers=2, out_dim=1).to(device)
Y_check = model_shape_probe(X_check)
assert Y_check.shape == (N_check, 1), f"Model output expected {(N_check,1)}, got {tuple(Y_check.shape)}"
assert Y_check.device == device, f"Model output on wrong device: {Y_check.device}"
print("[OK] Shape/dtype/device checks")


AssertionError: train_batch.mu_r expected shape (N,4), got (36000, 5)

In [9]:
# 2) Parameter encoding checks (unit tests)

def build_mu(case_meta: dict) -> np.ndarray:
    T0 = float(case_meta["T0"])
    q_type = int(case_meta["q_type"])
    q_amp = float(case_meta["q_amp"])
    q_freq = float(case_meta["q_freq"])
    if q_type == 0:
        q_freq = 0.0
    return np.array([T0, float(q_type), q_amp, q_freq], dtype=np.float32)

meta_const = {"T0": 293.15, "q_type": 0, "q_amp": 5000.0, "q_freq": 123.0}
meta_sin = {"T0": 293.15, "q_type": 1, "q_amp": 5000.0, "q_freq": 2.0}
mu_const = build_mu(meta_const)
mu_sin = build_mu(meta_sin)

assert np.array_equal(mu_const, np.array([293.15, 0.0, 5000.0, 0.0], dtype=np.float32)), f"build_mu const failed: {mu_const}"
assert np.array_equal(mu_sin, np.array([293.15, 1.0, 5000.0, 2.0], dtype=np.float32)), f"build_mu sin failed: {mu_sin}"
print("[OK] build_mu unit tests")


[OK] build_mu unit tests


In [10]:
# 3) Boundary heat flux function check (plot + asserts)

def q_right(t, mu):
    # mu = [T0, q_type, q_amp, q_freq]
    q_type = int(round(float(mu[1])))
    q_amp = float(mu[2])
    q_freq = float(mu[3])
    t_arr = np.asarray(t, dtype=np.float32)
    if q_type == 0:
        return np.full_like(t_arr, q_amp, dtype=np.float32)
    return (q_amp * np.sin(2.0 * np.pi * q_freq * t_arr)).astype(np.float32)


t_plot = np.linspace(0.0, 1.0, 400, dtype=np.float32)
mu_const_plot = np.array([293.15, 0.0, 2.0, 0.0], dtype=np.float32)
mu_sin_plot = np.array([293.15, 1.0, 2.0, 2.0], dtype=np.float32)  # integer periods over [0,1]

q_const = q_right(t_plot, mu_const_plot)
q_sin = q_right(t_plot, mu_sin_plot)

assert np.allclose(q_const, mu_const_plot[2], atol=1e-7), "q_right constant case is not constant q_amp"
assert np.var(q_const) < 1e-10, f"Constant q variance too high: {np.var(q_const)}"
assert np.var(q_sin) > 1e-4, "Sinusoidal q variance should be > 0"
assert np.isclose(np.max(np.abs(q_sin)), mu_sin_plot[2], rtol=5e-2, atol=5e-2), "Sinusoidal max amplitude not close to q_amp"
assert np.isclose(np.mean(q_sin), 0.0, atol=1e-2), f"Sinusoidal mean not ~0 over integer periods: {np.mean(q_sin)}"

plt.figure(figsize=(6.5, 3.8))
plt.plot(t_plot, q_const, label="q_type=0 (constant)")
plt.plot(t_plot, q_sin, label="q_type=1 (sinusoidal)")
plt.xlabel("t")
plt.ylabel("q_right")
plt.title("Boundary Flux Function Check")
plt.legend()
plt.tight_layout()
if "OUTDIR" in globals() and OUTDIR is not None:
    out_plot = OUTDIR / "q_right_verification.png"
    plt.savefig(out_plot, dpi=160)
    print(f"Saved q_right plot: {out_plot}")
else:
    plt.show()
plt.close()
print("[OK] q_right behavior checks")


Saved q_right plot: c:\Users\wscm13\OneDrive - Loughborough University\Part C\IDP\Individual Project\PINN\outputs\parametric\20260309_144024\q_right_verification.png
[OK] q_right behavior checks


In [11]:
# 4) Autograd correctness and 5) loss term wiring

def _grad(outputs, inputs):
    return torch.autograd.grad(
        outputs=outputs,
        inputs=inputs,
        grad_outputs=torch.ones_like(outputs),
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]


def _stack_input(xi, tau, mu):
    return torch.cat([xi, tau, mu], dim=1)


def pde_residual(model, xi, tau, mu):
    xi_r = xi.clone().detach().requires_grad_(True)
    tau_r = tau.clone().detach().requires_grad_(True)
    mu_r = mu.clone().detach().requires_grad_(False)
    theta = model(_stack_input(xi_r, tau_r, mu_r))
    dTdt = _grad(theta, tau_r)
    dTdx = _grad(theta, xi_r)
    d2Tdx2 = _grad(dTdx, xi_r)
    return dTdt - d2Tdx2, dTdx, dTdt, d2Tdx2


def bc_flux_from_mu(mu_bc, tau_bc, bc_time_scale):
    is_const = mu_bc[:, 1:2]
    is_sine = mu_bc[:, 2:3]
    q_amp_norm = mu_bc[:, 3:4]
    q_freq_norm = mu_bc[:, 4:5]

    q_amp = 0.5 * (q_amp_norm + 1.0) * (mu_stats["q_amp_max"] - mu_stats["q_amp_min"]) + mu_stats["q_amp_min"]
    q_freq = 0.5 * (q_freq_norm + 1.0) * (mu_stats["q_freq_max"] - mu_stats["q_freq_min"]) + mu_stats["q_freq_min"]
    t_phys = tau_bc * bc_time_scale
    q_const = q_amp
    q_sine = q_amp * torch.sin(2.0 * np.pi * q_freq * t_phys)
    return is_const * q_const + is_sine * q_sine


def loss_terms(model, batch):
    r, _, _, _ = pde_residual(model, batch.xi_r, batch.tau_r, batch.mu_r)
    Lpde = torch.mean(r ** 2)

    theta_ic_hat = model(_stack_input(batch.xi_ic, batch.tau_ic, batch.mu_ic))
    Lic = torch.mean((theta_ic_hat - batch.theta_ic) ** 2)

    xi_bc = batch.xi_bc.clone().detach().requires_grad_(True)
    tau_bc = batch.tau_bc.clone().detach().requires_grad_(True)
    mu_bc = batch.mu_bc.clone().detach().requires_grad_(False)
    theta_bc = model(_stack_input(xi_bc, tau_bc, mu_bc))
    dTdx_bc = _grad(theta_bc, xi_bc)
    q_bc = bc_flux_from_mu(mu_bc, tau_bc, batch.bc_time_scale)
    flux_target = -q_bc / batch.bc_flux_scale
    Lbc = torch.mean((dTdx_bc - flux_target) ** 2)

    theta_data_hat = model(_stack_input(batch.xi_data, batch.tau_data, batch.mu_data))
    Ldata = torch.mean((theta_data_hat - batch.theta_data) ** 2)
    return Lpde, Lic, Lbc, Ldata


N = 8
xg = train_batch.xi_r[:N].clone().detach().requires_grad_(True)
tg = train_batch.tau_r[:N].clone().detach().requires_grad_(True)
mug = train_batch.mu_r[:N].clone().detach().requires_grad_(False)
assert mug.requires_grad is False, "mu must not require grad"

probe = MLP(in_dim=2 + mug.shape[1], hidden=20, layers=2, out_dim=1).to(device)
Tg = probe(_stack_input(xg, tg, mug))
dTdx = _grad(Tg, xg)
dTdt = _grad(Tg, tg)
d2Tdx2 = _grad(dTdx, xg)

for nm, ten in [("dTdx", dTdx), ("dTdt", dTdt), ("d2Tdx2", d2Tdx2)]:
    assert ten is not None, f"{nm} is None"
    assert torch.isfinite(ten).all(), f"{nm} contains NaN/Inf"
print("[OK] Autograd derivative checks")

Lpde, Lic, Lbc, Ldata = loss_terms(probe, train_batch)
for nm, L in [("Lpde", Lpde), ("Lic", Lic), ("Lbc", Lbc), ("Ldata", Ldata)]:
    assert isinstance(L, torch.Tensor) and L.ndim == 0, f"{nm} must be scalar tensor"
    assert torch.isfinite(L).item(), f"{nm} not finite"

weights_test = LossWeights(w_pde=1.0, w_ic=1.0, w_bc=1.0, w_data=1.0)
Ltot = weights_test.w_pde * Lpde + weights_test.w_ic * Lic + weights_test.w_bc * Lbc + weights_test.w_data * Ldata
Lsum = Lpde + Lic + Lbc + Ldata
assert torch.isclose(Ltot, Lsum, rtol=1e-6, atol=1e-8), "Total loss is not equal to weighted sum"
print("[OK] Loss wiring checks")


[OK] Autograd derivative checks
[OK] Loss wiring checks


In [ ]:
# 6) Overfit-a-tiny-batch test and 7) case-conditioning test

def build_tiny_single_case_batch(case, device, seed=42):
    rng = np.random.default_rng(seed)
    mu = case["mu"].astype(np.float32)
    tau_max = float(np.max(case["nondim"]["tau"]))

    N_collocation = 128
    N_ic = 64
    N_bc = 64
    N_data = 128

    xi_r = rng.uniform(0.0, 1.0, size=(N_collocation, 1)).astype(np.float32)
    tau_r = rng.uniform(0.0, tau_max, size=(N_collocation, 1)).astype(np.float32)
    mu_r = np.repeat(mu.reshape(1, -1), N_collocation, axis=0)

    xi_ic_all = case["ic"]["xi_ic"].astype(np.float32)
    theta_ic_all = case["ic"]["theta_ic"].astype(np.float32)
    idx_ic = rng.choice(xi_ic_all.shape[0], size=N_ic, replace=True)
    xi_ic = xi_ic_all[idx_ic]
    tau_ic = np.zeros((N_ic, 1), dtype=np.float32)
    theta_ic = theta_ic_all[idx_ic]
    mu_ic = np.repeat(mu.reshape(1, -1), N_ic, axis=0)

    tau_bc_all = case["bc"]["tau_bc"].astype(np.float32).reshape(-1)
    idx_bc = rng.choice(tau_bc_all.shape[0], size=N_bc, replace=True)
    tau_bc = tau_bc_all[idx_bc].reshape(-1, 1).astype(np.float32)
    xi_bc = np.ones((N_bc, 1), dtype=np.float32)
    mu_bc = np.repeat(mu.reshape(1, -1), N_bc, axis=0)

    xi_data_all = case["interior"]["xi_data_all"].astype(np.float32)
    tau_data_all = case["interior"]["tau_data_all"].astype(np.float32)
    theta_data_all = case["interior"]["theta_data_all"].astype(np.float32)
    idx_d = rng.choice(xi_data_all.shape[0], size=N_data, replace=True)
    xi_data = xi_data_all[idx_d]
    tau_data = tau_data_all[idx_d]
    theta_data = theta_data_all[idx_d]
    mu_data = np.repeat(mu.reshape(1, -1), N_data, axis=0)

    tt = lambda a: torch.tensor(a, dtype=torch.float32, device=device)
    return ParamPINNBatch(
        xi_r=tt(xi_r), tau_r=tt(tau_r), mu_r=tt(mu_r),
        xi_ic=tt(xi_ic), tau_ic=tt(tau_ic), mu_ic=tt(mu_ic), theta_ic=tt(theta_ic),
        xi_bc=tt(xi_bc), tau_bc=tt(tau_bc), mu_bc=tt(mu_bc), bc_time_scale=tt(bc_time_scale), bc_flux_scale=tt(bc_flux_scale),
        xi_data=tt(xi_data), tau_data=tt(tau_data), mu_data=tt(mu_data), theta_data=tt(theta_data),
    )


def concat_param_batches(a: ParamPINNBatch, b: ParamPINNBatch) -> ParamPINNBatch:
    cat = lambda x, y: torch.cat([x, y], dim=0)
    return ParamPINNBatch(
        xi_r=cat(a.xi_r, b.xi_r), tau_r=cat(a.tau_r, b.tau_r), mu_r=cat(a.mu_r, b.mu_r),
        xi_ic=cat(a.xi_ic, b.xi_ic), tau_ic=cat(a.tau_ic, b.tau_ic), mu_ic=cat(a.mu_ic, b.mu_ic), theta_ic=cat(a.theta_ic, b.theta_ic),
        xi_bc=cat(a.xi_bc, b.xi_bc), tau_bc=cat(a.tau_bc, b.tau_bc), mu_bc=cat(a.mu_bc, b.mu_bc),
        bc_time_scale=cat(a.bc_time_scale, b.bc_time_scale), bc_flux_scale=cat(a.bc_flux_scale, b.bc_flux_scale),
        xi_data=cat(a.xi_data, b.xi_data), tau_data=cat(a.tau_data, b.tau_data), mu_data=cat(a.mu_data, b.mu_data), theta_data=cat(a.theta_data, b.theta_data),
    )


def rmse_data_local(model, batch):
    model.eval()
    with torch.no_grad():
        pred = model(torch.cat([batch.xi_data, batch.tau_data, batch.mu_data], dim=1))
        return float(torch.sqrt(torch.mean((pred - batch.theta_data) ** 2)).item())


def total_loss_local(model, batch, weights):
    Lpde, Lic, Lbc, Ldata = loss_terms(model, batch)
    Ltot = weights.w_pde * Lpde + weights.w_ic * Lic + weights.w_bc * Lbc + weights.w_data * Ldata
    return Ltot, {"Lpde": float(Lpde.item()), "Lic": float(Lic.item()), "Lbc": float(Lbc.item()), "Ldata": float(Ldata.item())}


tiny_case = train_cases[0]
tiny_batch = build_tiny_single_case_batch(tiny_case, device=device, seed=SEED)

_tiny_model = MLP(in_dim=2 + tiny_batch.mu_r.shape[1], hidden=20, layers=2, out_dim=1).to(device)
_tiny_weights = LossWeights(w_pde=1.0, w_ic=1.0, w_bc=1.0, w_data=1.0)
_tiny_opt = torch.optim.Adam(_tiny_model.parameters(), lr=1e-3)

L0, parts0 = total_loss_local(_tiny_model, tiny_batch, _tiny_weights)
rmse0 = rmse_data_local(_tiny_model, tiny_batch)

TINY_STEPS = 250
for s in range(1, TINY_STEPS + 1):
    _tiny_opt.zero_grad(set_to_none=True)
    L, _ = total_loss_local(_tiny_model, tiny_batch, _tiny_weights)
    L.backward()
    _tiny_opt.step()

L1, parts1 = total_loss_local(_tiny_model, tiny_batch, _tiny_weights)
rmse1 = rmse_data_local(_tiny_model, tiny_batch)

print(f"Tiny overfit loss: start={float(L0.item()):.4e}, end={float(L1.item()):.4e}, ratio={float(L1.item())/max(float(L0.item()),1e-12):.4f}")
print(f"Tiny overfit data RMSE: start={rmse0:.4e}, end={rmse1:.4e}")

ratio = float(L1.item()) / max(float(L0.item()), 1e-12)
if ratio >= 0.1:
    warnings.warn(
        "Tiny-batch loss did not drop by >=10x. Possible causes: bad scaling, wrong BC sign convention, or wrong mu encoding.",
        RuntimeWarning,
    )

# Conditioning proof model: deterministic synthetic two-mu fit.
# This is a focused functional proof that the network output changes with mu at fixed (x,t).

cond_probe = MLP(in_dim=2 + tiny_batch.mu_r.shape[1], hidden=20, layers=2, out_dim=1).to(device)
cond_opt = torch.optim.Adam(cond_probe.parameters(), lr=1e-3)

# Shared (x,t) grid
xi_eval = torch.linspace(0.0, 1.0, 25, device=device).reshape(-1, 1)
tau_eval = torch.linspace(0.0, 1.0, 25, device=device).reshape(-1, 1)
TT, XX = torch.meshgrid(tau_eval.reshape(-1), xi_eval.reshape(-1), indexing="ij")
xi_flat = XX.reshape(-1, 1)
tau_flat = TT.reshape(-1, 1)

# Two different mu vectors at the same (x,t)
mu_const_eval = torch.tensor([0.0, 1.0, 0.0, 0.0, -1.0], dtype=torch.float32, device=device).reshape(1, -1)
mu_sin_eval = torch.tensor([0.0, 0.0, 1.0, 0.0, 1.0], dtype=torch.float32, device=device).reshape(1, -1)
mu_const_grid = mu_const_eval.repeat(xi_flat.shape[0], 1)
mu_sin_grid = mu_sin_eval.repeat(xi_flat.shape[0], 1)

# Synthetic targets with explicit mu dependence to validate conditioning path.
y_const = 0.3 * xi_flat + 0.2 * tau_flat + 0.0
y_sin = 0.3 * xi_flat + 0.2 * tau_flat + 0.1 * torch.sin(2.0 * np.pi * tau_flat)

X_cond = torch.cat([
    torch.cat([xi_flat, tau_flat, mu_const_grid], dim=1),
    torch.cat([xi_flat, tau_flat, mu_sin_grid], dim=1),
], dim=0)
y_cond = torch.cat([y_const, y_sin], dim=0)

for _ in range(250):
    cond_opt.zero_grad(set_to_none=True)
    pred = cond_probe(X_cond)
    Lc = torch.mean((pred - y_cond) ** 2)
    Lc.backward()
    cond_opt.step()

cond_probe.eval()
with torch.no_grad():
    pred_const = cond_probe(torch.cat([xi_flat, tau_flat, mu_const_grid], dim=1))
    pred_sin = cond_probe(torch.cat([xi_flat, tau_flat, mu_sin_grid], dim=1))

mad = float(torch.mean(torch.abs(pred_const - pred_sin)).item())
print(f"Conditioning test mean |pred(mu_const)-pred(mu_sin)| = {mad:.4e}")
assert mad > 1e-4, "Parametric conditioning check failed: output is not sensitive to mu"
print("[OK] Tiny overfit + conditioning checks")


Tiny overfit loss: start=2.0619e+00, end=1.1426e-01, ratio=0.0554
Tiny overfit data RMSE: start=3.9808e-01, end=1.3009e-01
Conditioning test mean |pred(mu_const)-pred(mu_sin)| = 1.7512e-02
[OK] Tiny overfit + conditioning checks


## Training


In [14]:
TWOPI = 2.0 * np.pi


def _grad(outputs, inputs, create_graph: bool):
    return torch.autograd.grad(
        outputs=outputs,
        inputs=inputs,
        grad_outputs=torch.ones_like(outputs),
        create_graph=create_graph,
        retain_graph=True,
        only_inputs=True,
    )[0]


def _stack_input(xi, tau, mu):
    return torch.cat([xi, tau, mu], dim=1)


def pde_residual(model, xi, tau, mu, create_graph: bool = True):
    xi = xi.clone().detach().requires_grad_(True)
    tau = tau.clone().detach().requires_grad_(True)
    theta = model(_stack_input(xi, tau, mu))

    # Need first-derivative graph for second derivatives even in validation.
    theta_tau = _grad(theta, tau, create_graph=True)
    theta_x = _grad(theta, xi, create_graph=True)
    theta_xx = _grad(theta_x, xi, create_graph=create_graph)
    return theta_tau - theta_xx


def theta_x(model, xi, tau, mu, create_graph: bool = True):
    xi = xi.clone().detach().requires_grad_(True)
    tau = tau.clone().detach().requires_grad_(True)
    theta = model(_stack_input(xi, tau, mu))
    return _grad(theta, xi, create_graph=create_graph)


def bc_flux_from_mu(mu_bc, tau_bc, bc_time_scale):
    is_const = mu_bc[:, 1:2]
    is_sine = mu_bc[:, 2:3]

    q_amp_norm = mu_bc[:, 3:4]
    q_freq_norm = mu_bc[:, 4:5]

    q_amp_min = float(mu_stats["q_amp_min"])
    q_amp_max = float(mu_stats["q_amp_max"])
    q_freq_min = float(mu_stats["q_freq_min"])
    q_freq_max = float(mu_stats["q_freq_max"])

    q_amp = 0.5 * (q_amp_norm + 1.0) * (q_amp_max - q_amp_min) + q_amp_min
    q_freq = 0.5 * (q_freq_norm + 1.0) * (q_freq_max - q_freq_min) + q_freq_min

    t_phys = tau_bc * bc_time_scale
    q_const = q_amp
    q_sine = q_amp * torch.sin(TWOPI * q_freq * t_phys)
    q = is_const * q_const + is_sine * q_sine
    return q


def compute_losses_param(model, batch, weights, create_graph: bool = True):
    r = pde_residual(model, batch.xi_r, batch.tau_r, batch.mu_r, create_graph=create_graph)
    loss_pde = torch.mean(r ** 2)

    theta_ic_hat = model(_stack_input(batch.xi_ic, batch.tau_ic, batch.mu_ic))
    loss_ic = torch.mean((theta_ic_hat - batch.theta_ic) ** 2)

    theta_x_bc = theta_x(model, batch.xi_bc, batch.tau_bc, batch.mu_bc, create_graph=create_graph)
    q_bc = bc_flux_from_mu(batch.mu_bc, batch.tau_bc, batch.bc_time_scale)
    flux_target = -q_bc / batch.bc_flux_scale
    loss_bc = torch.mean((theta_x_bc - flux_target) ** 2)

    theta_data_hat = model(_stack_input(batch.xi_data, batch.tau_data, batch.mu_data))
    loss_data = torch.mean((theta_data_hat - batch.theta_data) ** 2)

    total = (
        weights.w_pde * loss_pde
        + weights.w_ic * loss_ic
        + weights.w_bc * loss_bc
        + weights.w_data * loss_data
    )

    logs = {
        "total": float(total.detach().cpu().item()),
        "pde": float(loss_pde.detach().cpu().item()),
        "ic": float(loss_ic.detach().cpu().item()),
        "bc": float(loss_bc.detach().cpu().item()),
        "data": float(loss_data.detach().cpu().item()),
    }
    return total, logs


def compute_losses_param_multi(model, batches, weights, create_graph: bool = True):
    totals = []
    logs_acc = {"total": 0.0, "pde": 0.0, "ic": 0.0, "bc": 0.0, "data": 0.0}
    for b in batches:
        total_b, logs_b = compute_losses_param(model, b, weights, create_graph=create_graph)
        totals.append(total_b)
        for k in logs_acc.keys():
            logs_acc[k] += float(logs_b[k])

    n = max(1, len(batches))
    total = torch.stack(totals).mean()
    logs = {k: (v / n) for k, v in logs_acc.items()}
    return total, logs


def rmse_data(model, batch):
    model.eval()
    with torch.no_grad():
        pred = model(_stack_input(batch.xi_data, batch.tau_data, batch.mu_data))
        mse = torch.mean((pred - batch.theta_data) ** 2).item()
    return float(np.sqrt(max(mse, 0.0)))


def sample_cases(cases, k, rng):
    if len(cases) == 0:
        return []
    k_eff = min(int(k), len(cases))
    idx = rng.choice(len(cases), size=k_eff, replace=False)
    return [cases[int(i)] for i in np.atleast_1d(idx)]


# BC sanity checks using physical q_amp/q_freq in mu
for case in train_cases + val_cases:
    if case["mu_raw"]["q_type_raw"] != "constant":
        tau_probe = torch.linspace(0.0, 1.0, 64, dtype=torch.float32, device=device).reshape(-1, 1)
        mu_probe = torch.tensor(np.repeat(case["mu"].reshape(1, -1), tau_probe.shape[0], axis=0), dtype=torch.float32, device=device)
        ts_probe = torch.full_like(tau_probe, float(case["time_scale"]))
        q_probe = bc_flux_from_mu(mu_probe, tau_probe, ts_probe)
        print(f"BC sine sanity case={case['case_id']}: q_min={float(q_probe.min().item()):.3e}, q_max={float(q_probe.max().item()):.3e}")
        break

for case in train_cases + val_cases:
    if case["mu_raw"]["q_type_raw"] == "constant":
        tau_probe = torch.linspace(0.0, 1.0, 64, dtype=torch.float32, device=device).reshape(-1, 1)
        mu_probe = torch.tensor(np.repeat(case["mu"].reshape(1, -1), tau_probe.shape[0], axis=0), dtype=torch.float32, device=device)
        ts_probe = torch.full_like(tau_probe, float(case["time_scale"]))
        q_probe = bc_flux_from_mu(mu_probe, tau_probe, ts_probe)
        q_span = float((q_probe.max() - q_probe.min()).item())
        print(f"BC const sanity case={case['case_id']}: q_span={q_span:.3e} (should be ~0)")
        break


model = MLP(in_dim=7, hidden=40, layers=8, out_dim=1).to(device)
weights = LossWeights(w_pde=1.0, w_ic=1.0, w_bc=1.0, w_data=1.0)

adam_steps = 50 if SHORT_TEST else 2000
adam_lr = 1e-3
lbfgs_max_iter = 50 if SHORT_TEST else 500

history = []
start_t = time.time()

cases_per_step = 1
rng = np.random.default_rng(42)
opt = torch.optim.Adam(model.parameters(), lr=adam_lr)
for step in range(1, adam_steps + 1):
    model.train()
    opt.zero_grad(set_to_none=True)

    chosen = sample_cases(train_cases, cases_per_step, rng)
    train_batches = build_parametric_batches(
        chosen,
        device=device,
        n_r_per_case=n_r_per_case,
        data_fraction=0.08 if SHORT_TEST else 0.25,
        seed=step,
        full_data=False,
    )

    sine_frac = float(np.mean([1.0 if c["mu_raw"]["q_type_raw"] != "constant" else 0.0 for c in chosen])) if chosen else 0.0
    loss, train_logs = compute_losses_param_multi(model, train_batches, weights, create_graph=True)
    loss.backward()
    opt.step()

    if step == 1 or step % 10 == 0 or step == adam_steps:
        _, val_logs = compute_losses_param_multi(model, val_batches, weights, create_graph=False)
        val_rmse = rmse_data(model, val_batch)
        history.append({
            "stage": "adam",
            "step": step,
            "train_total": train_logs["total"],
            "train_pde": train_logs["pde"],
            "train_ic": train_logs["ic"],
            "train_bc": train_logs["bc"],
            "train_data": train_logs["data"],
            "val_total": val_logs["total"],
            "val_pde": val_logs["pde"],
            "val_ic": val_logs["ic"],
            "val_bc": val_logs["bc"],
            "val_data": val_logs["data"],
            "val_rmse": val_rmse,
        })
        print(
            f"[Adam] step={step:4d} train={train_logs['total']:.3e} "
            f"(pde={train_logs['pde']:.2e}, ic={train_logs['ic']:.2e}, bc={train_logs['bc']:.2e}, data={train_logs['data']:.2e}) "
            f"val={val_logs['total']:.3e} val_rmse={val_rmse:.3e} sine_frac={sine_frac:.2f}"
        )


lbfgs_case_count = min(4, len(train_cases))
lbfgs_cases = sample_cases(train_cases, lbfgs_case_count, rng)
lbfgs_batches = build_parametric_batches(
    lbfgs_cases,
    device=device,
    n_r_per_case=n_r_per_case,
    data_fraction=0.08 if SHORT_TEST else 0.25,
    seed=999,
    full_data=False,
)

lbfgs = torch.optim.LBFGS(
    model.parameters(),
    lr=1.0,
    max_iter=lbfgs_max_iter,
    history_size=50,
    line_search_fn="strong_wolfe",
)


def closure():
    lbfgs.zero_grad(set_to_none=True)
    total, _ = compute_losses_param_multi(model, lbfgs_batches, weights, create_graph=True)
    total.backward()
    return total

print("[LBFGS] starting")
lbfgs.step(closure)
_, train_logs_final = compute_losses_param_multi(model, lbfgs_batches, weights, create_graph=False)
_, val_logs_final = compute_losses_param_multi(model, val_batches, weights, create_graph=False)
val_rmse_final = rmse_data(model, val_batch)
history.append({
    "stage": "lbfgs",
    "step": lbfgs_max_iter,
    "train_total": train_logs_final["total"],
    "train_pde": train_logs_final["pde"],
    "train_ic": train_logs_final["ic"],
    "train_bc": train_logs_final["bc"],
    "train_data": train_logs_final["data"],
    "val_total": val_logs_final["total"],
    "val_pde": val_logs_final["pde"],
    "val_ic": val_logs_final["ic"],
    "val_bc": val_logs_final["bc"],
    "val_data": val_logs_final["data"],
    "val_rmse": val_rmse_final,
})

elapsed = float(time.time() - start_t)
print(f"[LBFGS] done train={train_logs_final['total']:.3e} val={val_logs_final['total']:.3e} val_rmse={val_rmse_final:.3e}")


BC sine sanity case=offset_sine_1: q_min=0.000e+00, q_max=5.000e+03
BC const sanity case=const_10000: q_span=0.000e+00 (should be ~0)
[Adam] step=   1 train=9.378e-01 (pde=7.78e-02, ic=7.28e-03, bc=7.66e-01, data=8.69e-02) val=1.049e+00 val_rmse=3.452e-01 sine_frac=0.00
[Adam] step=  10 train=4.965e-01 (pde=2.79e-02, ic=6.56e-02, bc=3.53e-01, data=5.03e-02) val=1.027e+00 val_rmse=3.023e-01 sine_frac=0.00
[Adam] step=  20 train=1.908e-01 (pde=3.37e-02, ic=5.94e-02, bc=5.50e-02, data=4.27e-02) val=1.158e+00 val_rmse=3.760e-01 sine_frac=0.00
[Adam] step=  30 train=2.089e-01 (pde=2.67e-02, ic=6.53e-02, bc=9.20e-02, data=2.49e-02) val=9.805e-01 val_rmse=2.988e-01 sine_frac=0.00
[Adam] step=  40 train=1.924e-01 (pde=4.74e-03, ic=3.48e-02, bc=9.55e-02, data=5.74e-02) val=9.854e-01 val_rmse=3.003e-01 sine_frac=1.00
[Adam] step=  50 train=2.984e-01 (pde=8.91e-03, ic=8.24e-02, bc=1.76e-01, data=3.14e-02) val=9.593e-01 val_rmse=2.943e-01 sine_frac=1.00
[Adam] step=  60 train=2.768e-01 (pde=5.55e-

KeyboardInterrupt: 

## Outputs and Logging


In [ ]:
history_df = pd.DataFrame(history)
history_df.to_csv(OUTDIR / "loss_history.csv", index=False)

plt.figure(figsize=(7, 4))
if not history_df.empty:
    plt.plot(history_df.index, history_df["train_total"], label="train total")
    plt.plot(history_df.index, history_df["val_total"], label="val total")
plt.yscale("log")
plt.xlabel("Logged point")
plt.ylabel("Loss")
plt.title("Loss Curves")
plt.legend()
plt.tight_layout()
plt.savefig(OUTDIR / "loss_curves.png", dpi=160)
plt.close()

heldout_case = val_cases[0]
xi = heldout_case["nondim"]["xi"].astype(np.float32)
tau = heldout_case["nondim"]["tau"].astype(np.float32)
mu = heldout_case["mu"].astype(np.float32)

TT, XX = np.meshgrid(tau, xi, indexing="ij")
X_in = torch.tensor(
    np.column_stack([
        XX.reshape(-1, 1),
        TT.reshape(-1, 1),
        np.repeat(mu.reshape(1, -1), XX.size, axis=0),
    ]),
    dtype=torch.float32,
    device=device,
)

model.eval()
with torch.no_grad():
    theta_pred = model(X_in).cpu().numpy().reshape(len(tau), len(xi))

theta_ref = heldout_case["nondim"]["theta"].T
abs_err = np.abs(theta_pred - theta_ref)

T_left = float(heldout_case["physical"]["T_ref"])
dT = float(heldout_case["physical"]["dT_ref"])
T_pred = T_left + dT * theta_pred
T_ref = heldout_case["raw"]["T"].T

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
im0 = axes[0].imshow(T_ref, aspect="auto", origin="lower", extent=[xi.min(), xi.max(), tau.min(), tau.max()])
axes[0].set_title("Held-out T_ref")
axes[0].set_xlabel("xi")
axes[0].set_ylabel("tau")
plt.colorbar(im0, ax=axes[0])
im1 = axes[1].imshow(T_pred, aspect="auto", origin="lower", extent=[xi.min(), xi.max(), tau.min(), tau.max()])
axes[1].set_title("Held-out T_pred")
axes[1].set_xlabel("xi")
axes[1].set_ylabel("tau")
plt.colorbar(im1, ax=axes[1])
fig.savefig(OUTDIR / "heatmap_pred_vs_ref.png", dpi=160)
plt.close(fig)

plt.figure(figsize=(5.5, 4.2))
plt.imshow(abs_err, aspect="auto", origin="lower", extent=[xi.min(), xi.max(), tau.min(), tau.max()])
plt.title("Absolute Error Heatmap (theta)")
plt.xlabel("xi")
plt.ylabel("tau")
plt.colorbar()
plt.tight_layout()
plt.savefig(OUTDIR / "error_heatmap.png", dpi=160)
plt.close()

plt.figure(figsize=(5.5, 5.0))
plt.scatter(theta_ref.reshape(-1), theta_pred.reshape(-1), s=4, alpha=0.5)
mn = min(theta_ref.min(), theta_pred.min())
mx = max(theta_ref.max(), theta_pred.max())
plt.plot([mn, mx], [mn, mx], "k--", linewidth=1)
plt.xlabel("theta_ref")
plt.ylabel("theta_pred")
plt.title("Held-out Scatter")
plt.tight_layout()
plt.savefig(OUTDIR / "scatter_pred_vs_ref.png", dpi=160)
plt.close()

val_rmse = float(np.sqrt(np.mean((theta_pred - theta_ref) ** 2)))
num = float(np.linalg.norm(theta_pred - theta_ref))
den = float(np.linalg.norm(theta_ref))
relative_l2_error = float(num / den) if den > 0 else np.nan

train_probe_for_config = build_parametric_batch(
    sample_cases(train_cases, cases_per_step, np.random.default_rng(0)),
    device=device,
    n_r_per_case=n_r_per_case,
    data_fraction=0.08 if SHORT_TEST else 0.25,
    seed=0,
    full_data=False,
)

run_config = {
    "run_id": RUN_ID,
    "manifest": str(manifest),
    "train_case_ids": [c["case_id"] for c in train_cases],
    "heldout_case_ids": [c["case_id"] for c in val_cases],
    "short_test": SHORT_TEST,
    "architecture": "MLP(hidden=40, layers=4, activation=tanh)",
    "input_dim": 7,
    "output_dim": 1,
    "mu_definition": ["theta0", "is_const", "is_sine", "q_amp_norm", "q_freq_norm"],
    "mu_stats": mu_stats,
    "loss_weights": {
        "pde": weights.w_pde,
        "ic": weights.w_ic,
        "bc": weights.w_bc,
        "data": weights.w_data,
    },
    "sample_counts": {
        "train_collocation": int(train_probe_for_config.xi_r.shape[0]),
        "train_ic": int(train_probe_for_config.xi_ic.shape[0]),
        "train_bc": int(train_probe_for_config.xi_bc.shape[0]),
        "train_data": int(train_probe_for_config.xi_data.shape[0]),
        "val_collocation": int(val_batch.xi_r.shape[0]),
        "val_ic": int(val_batch.xi_ic.shape[0]),
        "val_bc": int(val_batch.xi_bc.shape[0]),
        "val_data": int(val_batch.xi_data.shape[0]),
    },
    "optimizer": {
        "stage1": "adam",
        "adam_lr": adam_lr,
        "adam_steps": adam_steps,
        "cases_per_step": cases_per_step,
        "stage2": "lbfgs",
        "lbfgs_max_iter": lbfgs_max_iter,
        "lbfgs_history_size": 50,
        "lbfgs_line_search_fn": "strong_wolfe",
        "lbfgs_case_count": int(min(4, len(train_cases))),
    },
}

with open(OUTDIR / "run_config.json", "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)

metrics_row = {
    "run_id": RUN_ID,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "architecture": run_config["architecture"],
    "loss_weights": json.dumps(run_config["loss_weights"]),
    "sample_counts": json.dumps(run_config["sample_counts"]),
    "final_total_loss": float(val_logs_final["total"]),
    "final_pde_loss": float(val_logs_final["pde"]),
    "final_ic_loss": float(val_logs_final["ic"]),
    "final_bc_loss": float(val_logs_final["bc"]),
    "final_data_loss": float(val_logs_final["data"]),
    "final_val_rmse": val_rmse,
    "relative_l2_error": relative_l2_error,
    "wall_clock_time_s": elapsed,
    "train_case_ids": ";".join(c["case_id"] for c in train_cases),
    "heldout_case_ids": ";".join(c["case_id"] for c in val_cases),
}

metrics_csv = ROOT / "outputs" / "parametric" / "metrics.csv"
metrics_df = pd.DataFrame([metrics_row])
if metrics_csv.exists():
    metrics_df.to_csv(metrics_csv, mode="a", header=False, index=False)
else:
    metrics_df.to_csv(metrics_csv, index=False)

print("Saved run_config:", OUTDIR / "run_config.json")
print("Saved metrics append file:", metrics_csv)
print("Saved plots:")
print("  ", OUTDIR / "loss_curves.png")
print("  ", OUTDIR / "heatmap_pred_vs_ref.png")
print("  ", OUTDIR / "error_heatmap.png")
print("  ", OUTDIR / "scatter_pred_vs_ref.png")
print(f"Held-out val_rmse={val_rmse:.4e}, relative_l2_error={relative_l2_error:.4e}")


Saved run_config: c:\Users\wscm13\OneDrive - Loughborough University\Part C\IDP\Individual Project\PINN\outputs\parametric\20260223_150339\run_config.json
Saved metrics append file: c:\Users\wscm13\OneDrive - Loughborough University\Part C\IDP\Individual Project\PINN\outputs\parametric\metrics.csv
Saved plots:
   c:\Users\wscm13\OneDrive - Loughborough University\Part C\IDP\Individual Project\PINN\outputs\parametric\20260223_150339\loss_curves.png
   c:\Users\wscm13\OneDrive - Loughborough University\Part C\IDP\Individual Project\PINN\outputs\parametric\20260223_150339\heatmap_pred_vs_ref.png
   c:\Users\wscm13\OneDrive - Loughborough University\Part C\IDP\Individual Project\PINN\outputs\parametric\20260223_150339\error_heatmap.png
   c:\Users\wscm13\OneDrive - Loughborough University\Part C\IDP\Individual Project\PINN\outputs\parametric\20260223_150339\scatter_pred_vs_ref.png
Held-out val_rmse=3.1083e-01, relative_l2_error=7.6345e-01
